In [0]:
dbutils.widgets.text("catalog","fk_1361")
catalog = dbutils.widgets.get("catalog")

In [0]:
from pyspark.sql import functions as F

In [0]:
df = spark.read.table(f"{catalog}.bronze.products_bronze")

In [0]:
df.display()

In [0]:
def standardize(df):
    return df.withColumns({
        "product_id" : F.trim(F.upper(F.col("product_id"))),
        "product_name" : F.trim(F.initcap(F.col("product_name"))),
        "brand" : F.trim(F.initcap(F.col("brand"))),
        "is_active" : F.trim(F.col("is_active")).cast("boolean"),
        "mrp" : F.col("mrp").cast("double"),
        "selling_price" : F.col("selling_price").cast("double"),
        "snapshot_date" : F.to_date(F.col("snapshot_date"), "yyyy-MM-dd")
    })

In [0]:
df_standard = standardize(df)

In [0]:
df_cleaned = df_standard.dropDuplicates().dropna()
df_cleaned.write.mode("overwrite").saveAsTable(f"{catalog}.silver.products_cleaned")

In [0]:
df_qurantine = df_standard.filter(F.col("product_id").isNull() | F.col("snapshot_date").isNull())
df_qurantine.write.mode("overwrite").saveAsTable(f"{catalog}.silver.products_qurantine")